# Xây dựng tool tự động crawl new từ google

## 0. Cài đặt và kiểm tra thư viện

In [ ]:
!pip install requests beautifulsoup4 lxml openpyxl googlenewsdecoder Pillow matplotlib

In [ ]:
import sys, importlib

print(sys.version)
for pkg in ["requests","bs4","lxml","openpyxl","PIL","googlenewsdecoder"]:
    try: importlib.import_module(pkg); print(f"✓ {pkg}")
    except Exception as e: print(f"✗ {pkg}: {e}")

## 1. Hằng số toàn cục

**Vai trò:** tập trung cấu hình dùng chung, tránh hardcode rải rác.

**Nhiệm vụ:** `HEADERS` giả Chrome, `TIMEOUT=15s`, `PLACEHOLDER_HOSTS` để `images` lọc logo Google.

In [ ]:
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}
TIMEOUT = 10 # TIMEOUT cho MỘT request HTTP đơn lẻ
DEFAULT_LANG = "vi"
DEFAULT_COUNTRY = "VN"
DEFAULT_WORKERS = 5
POOL_CONNECTIONS = 20
POOL_MAXSIZE = 20
DEFAULT_PROXY: str | None = None
PLACEHOLDER_HOSTS = ("googleusercontent.com", "gstatic.com", "google.com/images", "gnews/logo", "lh3.googleusercontent")

USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_6_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.6 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:130.0) Gecko/20100101 Firefox/130.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.6 Mobile/15E148 Safari/604.1",
]


def get_random_user_agent() -> str:
    """Return a random modern user agent string."""
    return random.choice(USER_AGENTS)


## 2. Các hàm xử lý URL

### 2.1. Hàm kiểm tra trạng thái của URL

**Vai trò:** một điểm duy nhất để GET publisher URL, phân loại mã HTTP.

**Nhiệm vụ:** trả `(html, code, "alive 200"/"dead 401"/...)`

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

_DEAD_CODES = {401, 403, 404, 410, 429, 500, 502, 503}


def create_session(
    proxy: str | None = None,
    pool_size: int = POOL_MAXSIZE,
    max_retries: int = 3,
    backoff: float = 0.5,
) -> requests.Session:
    """Create a configured requests.Session with connection pooling, proxy, and retry backoff."""
    session = requests.Session()
    session.headers.update(HEADERS)
    if proxy:
        session.proxies = {
            "http": proxy,
            "https": proxy,
        }
    adapter = HTTPAdapter(
        pool_connections=pool_size,
        pool_maxsize=pool_size,
        max_retries=Retry(
            total=max_retries,
            backoff_factor=backoff,
            status_forcelist=[429, 500, 502, 503, 504],
            raise_on_status=False,
        ),
    )
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def fetch_with_status(
    url: str,
    session: requests.Session | None = None,
    timeout: int = TIMEOUT,
    rotate_ua: bool = False,
) -> tuple[str | None, int | None, str]:
    """GET url, return (html_text, status_code, alive_status)."""
    headers = dict(HEADERS)
    if rotate_ua:
        headers["User-Agent"] = get_random_user_agent()

    try:
        if session is not None:
            req_headers = {"User-Agent": headers["User-Agent"]} if rotate_ua else None
            resp = session.get(url, headers=req_headers, timeout=timeout)
        else:
            resp = requests.get(url, headers=headers, timeout=timeout)
    except requests.exceptions.Timeout:
        return None, None, "timeout"
    except requests.exceptions.ConnectionError:
        return None, None, "connection_error"
    except Exception as e:
        return None, None, f"error:{type(e).__name__}"

    code = resp.status_code
    if code == 200 and resp.text:
        return resp.text, code, f"alive {code}"
    if code in _DEAD_CODES:
        return None, code, f"dead {code}"
    return None, code, f"http_{code}"


### 2.2. Hàm tìm News qua **Google News RSS**

**Vai trò:** discovery nhanh, không cần browser.

**Nhiệm vụ:** `hl=vi&gl=VN&ceid=VN:vi` → parse XML `item/title/link/pubDate/source` → `snippet` 500 ký tự.

In [ ]:
import html, xml.etree.ElementTree as ET
from urllib.parse import quote_plus
from bs4 import BeautifulSoup

def fetch_rss(
    keyword: str,
    max_articles: int,
    lang: str = DEFAULT_LANG,
    country: str = DEFAULT_COUNTRY,
    session: requests.Session | None = None,
    timeout: int = TIMEOUT,
):
    q = quote_plus(keyword)
    ceid = f"{country}:{lang}"
    url = f"https://news.google.com/rss/search?q={q}&hl={lang}&gl={country}&ceid={ceid}"
    if session is not None:
        resp = session.get(url, timeout=timeout)
    else:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    articles = []
    for it in root.findall(".//item")[:max_articles]:
        title = (it.findtext("title") or "").strip()
        link = (it.findtext("link") or "").strip()  # google CBM
        pub = (it.findtext("pubDate") or "").strip()
        desc_raw = it.findtext("description") or ""
        snippet = BeautifulSoup(html.unescape(desc_raw), "html.parser").get_text(" ", strip=True)[:500]
        source_el = it.find("source")
        source = (source_el.text.strip() if source_el is not None and source_el.text else "")
        if not source and " - " in title:
            source = title.rsplit(" - ", 1)[-1]
        articles.append({"title": title, "source": source, "google_url": link, "url": link, "published": pub, "snippet": snippet})
    return articles


### 2.3. Hàm giải mã CBM để lấy URL gốc (Multi-strategy Fallback)

**Vai trò:** biến `news.google.com/rss/articles/CBM...` thành `https://publisher/...` để fetch thật.

**Nhiệm vụ:**
1. Thử `gnewsdecoder(google_url, interval=0.1, proxy=proxy)`
2. Fallback 1: Gửi HTTP GET theo dõi Redirects (`allow_redirects=True`)
3. Fallback 2: Bóc tách meta refresh / noscript anchor từ trang HTML của Google.


In [ ]:
try:
    from googlenewsdecoder import gnewsdecoder
except ImportError:
    gnewsdecoder = None

from bs4 import BeautifulSoup


def _extract_url_from_html(html_text: str) -> str | None:
    """Extract destination URL from Google redirect HTML page."""
    soup = BeautifulSoup(html_text, "html.parser")
    # 1. Meta refresh
    for meta in soup.find_all("meta", attrs={"http-equiv": lambda x: x and x.lower() == "refresh"}):
        content = meta.get("content", "")
        if "url=" in content.lower():
            target = content.lower().split("url=")[-1].split(";")[0].strip("'\" ")
            if target.startswith("http") and "google.com" not in target:
                return target
    # 2. Noscript anchors
    for ns in soup.find_all("noscript"):
        for a in ns.find_all("a"):
            href = (a.get("href") or "").strip()
            if href.startswith("http") and "google.com" not in href:
                return href
    # 3. Data attributes / data-n-au
    for el in soup.find_all(attrs={"data-n-au": True}):
        u = el["data-n-au"].strip()
        if u.startswith("http") and "google.com" not in u:
            return u
    # 4. Fallback: check any external link inside main containers
    for a in soup.find_all("a"):
        href = (a.get("href") or "").strip()
        if href.startswith("http") and not any(h in href for h in ("google.com", "gstatic.com", "googleusercontent.com")):
            return href
    return None


def decode_publisher_url(
    google_url: str,
    session: requests.Session | None = None,
    proxy: str | None = None,
    timeout: int = TIMEOUT,
) -> tuple[str | None, str]:
    """
    Multi-strategy decoder: CBM google_url -> publisher_url.
    1. googlenewsdecoder (Primary)
    2. HTTP Redirect Follow (Fallback 1)
    3. HTML Meta Refresh / Anchor Extraction (Fallback 2)
    """
    if not google_url or "news.google.com" not in google_url:
        return None, "not_google_url"

    # Strategy 1: googlenewsdecoder
    if gnewsdecoder is not None:
        try:
            res = gnewsdecoder(google_url, interval=0.1, proxy=proxy)
            if isinstance(res, dict) and res.get("status") and res.get("decoded_url"):
                decoded = res["decoded_url"]
                if decoded and "google.com" not in decoded:
                    return decoded, "decoded"
        except Exception:
            pass

    # Strategy 2: HTTP Follow Redirect
    client = session or requests
    try:
        req_kwargs: dict = {"timeout": timeout, "allow_redirects": True}
        if session is None:
            req_kwargs["headers"] = HEADERS
            if proxy:
                req_kwargs["proxies"] = {"http": proxy, "https": proxy}

        resp = client.get(google_url, **req_kwargs)
        final_url = resp.url
        if final_url and "google.com" not in final_url and "google." not in final_url:
            return final_url, "redirect_followed"

        # Strategy 3: HTML Parser for meta refresh / embedded links
        if resp.text:
            extracted = _extract_url_from_html(resp.text)
            if extracted:
                return extracted, "html_extracted"

    except Exception as e:
        return None, f"fallback_error:{type(e).__name__}"

    return None, "decode_failed"


## 3. Các hàm xử lý phần hình ảnh

### 3.1. Hàm sử lý ảnh thừa, ảnh rác

**Vai trò:** loại ảnh logo Google/gstatic trước khi tải, tránh nhầm placeholder thành ảnh bài.

In [ ]:
def is_placeholder_url(url: str) -> bool:
    return any(h in url.lower() for h in PLACEHOLDER_HOSTS)

### 3.2. Hàm tìm **og:image**

**Nhiệm vụ:** quét `<meta property="og:image"/"twitter:image">`, trả list URL sạch.

In [ ]:
from urllib.parse import urljoin

def _meta_candidates(soup, base_url: str) -> list[str]:
    out: list[str] = []
    for tag in soup.find_all("meta"):
        prop = (tag.get("property", "") or tag.get("name", "")).strip().lower()
        if prop not in ("og:image", "og:image:secure_url", "twitter:image", "twitter:image:src"):
            continue
        content = (tag.get("content") or "").strip()
        if not content or is_placeholder_url(content):
            continue
        out.append(urljoin(base_url, content))
    return out

### 3.3. Hàm tìm link **image_src**

**Nhiệm vụ:** bắt `<link rel="image_src">` ít gặp nhưng chuẩn SEO.

In [ ]:
def _link_candidates(soup, base_url: str) -> list[str]:
    out: list[str] = []
    for tag in soup.find_all("link", rel=lambda x: x and "image_src" in x.lower() if isinstance(x, str) else False):
        href = (tag.get("href") or "").strip()
        if not href or is_placeholder_url(href):
            continue
        out.append(urljoin(base_url, href))
    return out

### 3.4. Hàm tìm **picture/source**

**Nhiệm vụ:** xử lý `<picture><source srcset>` (responsive), lấy URL đầu.

In [ ]:
def _picture_candidates(soup, base_url: str) -> list[str]:
    out: list[str] = []
    for source in soup.find_all("source"):
        srcset = source.get("srcset", "") or source.get("data-srcset", "")
        if not srcset:
            continue
        first = srcset.split(",")[0].strip().split()[0]
        if not first or first.startswith("data:") or is_placeholder_url(first):
            continue
        out.append(urljoin(base_url, first))
    return out

### 3.5. Hàm tìm thẻ `<img>`

**Nhiệm vụ:** quét `<img>` với `src|data-src|srcset`, lọc `80×80` spacer, `spinner/logo`, trả tới 8 URL.

In [ ]:
_JUNK = ("spinner", "pixel", "transparent", "1x1", "avatar")

def _img_candidates(soup, base_url: str) -> list[str]:
    out: list[str] = []
    for img in soup.find_all("img"):
        src = (img.get("src") or img.get("data-src") or img.get("data-original") or img.get("data-lazy-src") or "").strip()
        if not src:
            srcset = img.get("srcset", "") or img.get("data-srcset", "")
            if srcset:
                parts = [p.strip().split()[0] for p in srcset.split(",") if p.strip()]
                if parts:
                    src = parts[-1]
        if not src or src.startswith("data:") or is_placeholder_url(src):
            continue
        w, h = img.get("width", ""), img.get("height", "")
        try:
            if w and h and int(w) < 80 and int(h) < 80:
                continue
        except ValueError:
            pass
        low = src.lower()
        if any(x in low for x in _JUNK):
            continue
        if "logo" in low and "cdn" not in low:
            continue
        out.append(urljoin(base_url, src))
        if len(out) >= 8:
            break
    return out

### 3.6. Hàm tổng hợp ảnh

**Vai trò:** facade duy nhất cho 4 helper trên, trả tối đa 2 URL.

**Nhiệm vụ:** `candidates = [*_meta(), *_link(), *_picture(), *_img()]` → `dict.fromkeys` dedupe, lọc `http`.

In [ ]:
def extract_image_urls(page_url: str, html_text: str, base_url: str) -> list[str]:
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html_text, "lxml")
    candidates = [
        *_meta_candidates(soup, base_url),
        *_link_candidates(soup, base_url),
        *_picture_candidates(soup, base_url),
        *_img_candidates(soup, base_url),
    ]
    uniq = list(dict.fromkeys(u.split("#")[0].strip().rstrip('",\'') for u in candidates))
    uniq = [u for u in uniq if u.startswith("http") and not is_placeholder_url(u)]
    return uniq[:2]

### 3.7. Hàm tải ảnh về

**Vai trò:** tải từng URL, kiểm tra magic bytes + `Content-Type`, bỏ `<1KB`. Mặc định Tool sẽ lưu vào `output/images/`.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse

_VALID_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".gif")

def _download_single(
    url: str,
    dest_dir: Path,
    filename: str,
    session: requests.Session | None = None,
    timeout: int = TIMEOUT,
) -> Path | None:
    try:
        if session is not None:
            resp = session.get(url, timeout=timeout, stream=True)
        else:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, stream=True)
        if resp.status_code != 200:
            return None
        data = resp.content
        if len(data) < 1024:
            return None
        ctype = resp.headers.get("Content-Type", "")
        is_image = data.startswith((b"\xff\xd8", b"\x89PNG", b"GIF8", b"RIFF")) or "image" in ctype or "octet-stream" in ctype
        if not is_image and "text/html" in ctype:
            return None
        ext = Path(urlparse(url).path).suffix.lower()
        if ext == ".jpeg":
            ext = ".jpg"
        elif ext not in _VALID_EXTS:
            if (data.startswith(b"RIFF") and b"WEBP" in data[:16]) or "webp" in ctype:
                ext = ".webp"
            elif data.startswith(b"\x89PNG") or "png" in ctype:
                ext = ".png"
            elif data.startswith(b"GIF8") or "gif" in ctype:
                ext = ".gif"
            else:
                ext = ".jpg"
        fpath = dest_dir / f"{filename}{ext}"
        fpath.write_bytes(data)
        return fpath
    except Exception:
        return None

def download_images(
    image_urls: list[str],
    dest_dir: Path,
    prefix: str,
    session: requests.Session | None = None,
    timeout: int = TIMEOUT,
) -> list[Path]:
    """Download image URLs concurrently. Returns saved paths in candidate order."""
    if not image_urls:
        return []
    dest_dir.mkdir(parents=True, exist_ok=True)
    if len(image_urls) == 1:
        res = _download_single(image_urls[0], dest_dir, f"{prefix}_01", session=session, timeout=timeout)
        return [res] if res else []

    with ThreadPoolExecutor(max_workers=min(len(image_urls), 4)) as pool:
        futures = [
            pool.submit(_download_single, url, dest_dir, f"{prefix}_{idx:02d}", session, timeout)
            for idx, url in enumerate(image_urls, 1)
        ]
        results = [f.result() for f in futures]
    return [p for p in results if p is not None]


## 4. Các hàm xử lý file Excel

### 4.1. Hàm tạo style sheet chung

**Nhiệm vụ:** bold header, freeze `A2`, autofilter, `column width` 10 cột, `fitToPage`.

In [ ]:
from openpyxl.packaging.manifest import mimetypes as openpyxl_mimetypes
openpyxl_mimetypes.add_type("image/webp", ".webp")

from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

_HEADERS = ["#", "Title", "Source", "Google News URL", "Publisher URL", "Published", "Snippet", "Preview", "Images", "Status"]
_WIDTHS = [4, 42, 14, 32, 32, 18, 42, 16, 28, 30]

def _style_header(ws):
    for c in ws[1]: c.font = Font(bold=True)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(_HEADERS))}1"
    for idx, w in enumerate(_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(idx)].width = w
    if ws.sheet_properties.pageSetUpPr:
        ws.sheet_properties.pageSetUpPr.fitToPage = True

### 4.2. Hàm sử lý link

**Nhiệm vụ:** `Publisher URL` (col 5), `Google News URL` (col 4), `Images` (col 9) → `Hyperlink` style.

In [ ]:
def _add_hyperlinks(ws, row_idx: int, art: dict, rel: str):
    try:
        if link := art.get("publisher_url") or art.get("google_url", ""):
            ws.cell(row=row_idx, column=5).hyperlink = link
            ws.cell(row=row_idx, column=5).style = "Hyperlink"
        if g_link := art.get("google_url", ""):
            ws.cell(row=row_idx, column=4).hyperlink = g_link
            ws.cell(row=row_idx, column=4).style = "Hyperlink"
        if rel:
            first = rel.split(";")[0].strip()
            if first:
                ws.cell(row=row_idx, column=9).hyperlink = first
                ws.cell(row=row_idx, column=9).style = "Hyperlink"
    except: pass

### 4.3. Hàm nhúng Thumbnail

**Nhiệm vụ:** lấy `saved_images[0]`, `webp → PNG BytesIO` (thumbnail 160×120), `add_image` vào `H{row}` 100×75, row height 60.

In [ ]:
def _embed_preview(ws, row_idx: int, thumb_path):
    try:
        from io import BytesIO
        from openpyxl.drawing.image import Image as XLImage
        from PIL import Image as PILImage
    except: return
    if not thumb_path.exists(): return
    try:
        with PILImage.open(thumb_path) as im:
            im.thumbnail((160, 120))
            bio = BytesIO()
            if im.mode in ("RGBA", "LA"):
                bg = PILImage.new("RGB", im.size, (255, 255, 255))
                bg.paste(im, mask=im.split()[-1] if im.mode == "RGBA" else None)
                bg.save(bio, "PNG")
            else:
                im.convert("RGB").save(bio, "PNG")
            bio.seek(0)
            xl = XLImage(bio)
            xl.width = 100
            xl.height = 75
            ws.add_image(xl, f"H{row_idx}")
            ws.row_dimensions[row_idx].height = 60
    except: pass

### 4.4. Hàm gộp và xuất file

**Vai trò:** facade cho 3 helper trên, tạo `output/articles.xlsx` với 10 cột, wrap `Snippet`.

In [ ]:
from openpyxl import Workbook
from pathlib import Path

def export_excel(articles, output_dir: Path, images_dir_name="images"):
    xlsx = output_dir / "articles.xlsx"; wb = Workbook(); ws = wb.active; ws.title="Articles"
    ws.append(_HEADERS); _style_header(ws)
    for i, art in enumerate(articles, 1):
        imgs = art.get("saved_images", [])
        rel = "; ".join(f"{images_dir_name}/{p.name}" for p in imgs) if imgs and isinstance(imgs[0], Path) else "; ".join(str(p) for p in imgs) if imgs else ""
        ws.append([i, art.get("title",""), art.get("source",""), art.get("google_url",""), art.get("publisher_url",""), art.get("published",""), art.get("snippet",""), "", rel, art.get("status","")])
        row = i+1; _add_hyperlinks(ws, row, art, rel)
        if imgs: _embed_preview(ws, row, Path(imgs[0]))
        else: ws.row_dimensions[row].height = 15
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=7, max_col=7):
        for c in row: c.alignment = c.alignment.copy(wrap_text=True, vertical="top")
    wb.save(xlsx); return xlsx

def export_excel_from_jsonl(jsonl_path: str | Path, output_dir: str | Path | None = None) -> Path:
    jpath = Path(jsonl_path)
    out_dir = Path(output_dir) if output_dir else jpath.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    articles = []
    with open(jpath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            art = json.loads(line)
            if raw_imgs := art.get("saved_images"):
                art["saved_images"] = [Path(p) for p in raw_imgs]
            articles.append(art)
    return export_excel(articles, out_dir)


### 4.5. Hàm xử lý **Early Return**

**Vai trò:** `return` sớm nếu fail.

**Nhiệm vụ:** `decode → fetch → extract → download`, trả `status` string, ghi `publisher_url/saved_images` vào `art`.

In [ ]:
def _process_one(
    art: dict,
    prefix: str,
    images_dir: Path,
    session: requests.Session | None = None,
    proxy: str | None = None,
    rotate_ua: bool = False,
    timeout: int = TIMEOUT,
) -> str:
    """Process single article. Early returns, no nested ladder. Returns status."""
    google_url = art.get("google_url", "")
    if not google_url:
        return "no_google_url"

    publisher_url, decode_status = decode_publisher_url(google_url, session=session, proxy=proxy, timeout=timeout)
    if not publisher_url:
        return f"decode_failed ({decode_status})"
    art["publisher_url"] = publisher_url
    art["url"] = publisher_url

    html, code, alive_status = fetch_with_status(publisher_url, session=session, timeout=timeout, rotate_ua=rotate_ua)
    art["alive"] = alive_status
    art["http_code"] = code
    if html is None:
        return alive_status

    urls = extract_image_urls(publisher_url, html, publisher_url)
    if not urls:
        return f"{alive_status} | no_images_found"

    saved = download_images(urls, images_dir, prefix, session=session, timeout=timeout)
    art["saved_images"] = saved
    if not saved:
        return f"{alive_status} | download_failed | {len(urls)} candidates"
    return f"{alive_status} | ok | {len(saved)} image(s)"


## 5. Điều phối

**Vai trò:** điều phối toàn pipeline, đếm `decoded/alive/with_images` phẳng bằng `sum`.

In [ ]:
import json
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

_checkpoint_lock = threading.Lock()
_ui_lock = threading.Lock()


def _append_checkpoint(jsonl_path: Path, art: dict) -> None:
    """Append single article record to JSONL checkpoint file (thread-safe)."""
    record = dict(art)
    if imgs := record.get("saved_images"):
        record["saved_images"] = [str(p) for p in imgs]
    with _checkpoint_lock:
        with open(jsonl_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _load_completed_checkpoint(jsonl_path: Path) -> dict[str, dict]:
    """Load existing checkpoint into a mapping of google_url -> article dict."""
    if not jsonl_path.exists():
        return {}
    completed = {}
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                key = data.get("google_url") or data.get("url") or data.get("title")
                if key:
                    if imgs := data.get("saved_images"):
                        data["saved_images"] = [Path(p) for p in imgs]
                    completed[key] = data
            except Exception:
                continue
    return completed


def crawl_google_news(
    keyword: str,
    max_articles: int = 20,
    output_dir: str | Path = "output",
    *,
    workers: int = DEFAULT_WORKERS,
    proxy: str | None = None,
    rotate_ua: bool = False,
    lang: str = DEFAULT_LANG,
    country: str = DEFAULT_COUNTRY,
    timeout: int = TIMEOUT,
    resume: bool = True,
    show_progress: bool = True,
):
    if not keyword or not keyword.strip():
        raise ValueError("keyword must be non-empty")
    if max_articles <= 0:
        raise ValueError("max_articles must be > 0")
    if workers <= 0:
        raise ValueError("workers must be > 0")

    keyword = keyword.strip()
    output_dir = Path(output_dir)
    images_dir = output_dir / "images"
    jsonl_path = output_dir / "articles.jsonl"
    output_dir.mkdir(parents=True, exist_ok=True)
    images_dir.mkdir(parents=True, exist_ok=True)
    extra_info = []
    if proxy:
        extra_info.append(f"proxy={proxy}")
    if rotate_ua:
        extra_info.append("rotate_ua=True")
    extra_str = f" ({", ".join(extra_info)})" if extra_info else ""
    print(f"[crawl] keyword={keyword!r} max={max_articles} workers={workers} lang={lang} country={country}{extra_str} output={output_dir}")
    print("[crawl] Press Ctrl+C to stop safely at any time.")

    if not resume and jsonl_path.exists():
        jsonl_path.unlink()

    completed_map = _load_completed_checkpoint(jsonl_path) if resume else {}
    if completed_map:
        print(f"[crawl] loaded {len(completed_map)} previously completed article(s) from checkpoint")

    interrupted = False
    articles: list[dict] = []
    pool_size = max(POOL_MAXSIZE, workers * 4)

    with create_session(proxy=proxy, pool_size=pool_size) as session:
        try:
            articles = fetch_rss(keyword, max_articles, lang=lang, country=country, session=session, timeout=timeout)
        except KeyboardInterrupt:
            print("[crawl] Stop requested via Ctrl+C during RSS discovery.")
            xlsx = export_excel([], output_dir)
            return {
                "keyword": keyword,
                "discovered": 0,
                "processed": 0,
                "decoded": 0,
                "alive": 0,
                "with_images": 0,
                "saved_images": 0,
                "interrupted": True,
                "xlsx": str(xlsx),
                "jsonl": str(jsonl_path),
                "output_dir": str(output_dir),
                "images_dir": str(images_dir),
            }

        print(f"[crawl] discovered {len(articles)} articles")
        if not articles:
            xlsx = export_excel([], output_dir)
            return {
                "keyword": keyword,
                "discovered": 0,
                "processed": 0,
                "decoded": 0,
                "alive": 0,
                "with_images": 0,
                "saved_images": 0,
                "interrupted": False,
                "xlsx": str(xlsx),
                "jsonl": str(jsonl_path),
                "output_dir": str(output_dir),
                "images_dir": str(images_dir),
            }

        for art in articles:
            art["status"] = ""
            art["saved_images"] = []
            art["publisher_url"] = ""

        try:
            from tqdm.auto import tqdm as _tqdm
            _has_tqdm = True
        except ImportError:
            _has_tqdm = False
            _tqdm = None

        use_bar = show_progress and _has_tqdm
        pbar = _tqdm(total=len(articles), desc=keyword[:30], unit=" bài", ncols=90) if use_bar else None

        def _report_progress(idx: int, art: dict) -> None:
            with _ui_lock:
                title = (art.get("title") or "").strip()
                if len(title) > 80:
                    title = title[:77] + "..."
                source = art.get("source", "")
                status = art.get("status", "")
                line = f"{idx:3d}. {title} — {source} [{status}]"
                if use_bar:
                    try:
                        _tqdm.write(line)
                    except Exception:
                        print(line)
                    if pbar is not None:
                        try:
                            pbar.update(1)
                            alive_cnt = sum(1 for a in articles if a.get("http_code") == 200)
                            ok_cnt = sum(1 for a in articles if " | ok |" in a.get("status", ""))
                            pbar.set_postfix(alive=alive_cnt, ok=ok_cnt, refresh=False)
                        except Exception:
                            pass
                else:
                    print(line)

        def _worker_task(idx: int, art: dict) -> None:
            g_url = art.get("google_url", "")
            if resume and g_url in completed_map:
                prev = completed_map[g_url]
                art.update(prev)
                _report_progress(idx, art)
                return

            art["status"] = _process_one(
                art,
                f"article_{idx:05d}",
                images_dir,
                session=session,
                proxy=proxy,
                rotate_ua=rotate_ua,
                timeout=timeout,
            )
            _append_checkpoint(jsonl_path, art)
            _report_progress(idx, art)

        executor = ThreadPoolExecutor(max_workers=max(1, workers))
        try:
            futures = [
                executor.submit(_worker_task, idx, art)
                for idx, art in enumerate(articles, 1)
            ]
            for f in futures:
                f.result()
        except KeyboardInterrupt:
            interrupted = True
            print("[crawl] Stop requested (Ctrl+C). Cancelling pending tasks and saving progress...")
            executor.shutdown(wait=False, cancel_futures=True)
        finally:
            if use_bar and pbar is not None:
                try:
                    pbar.close()
                except Exception:
                    pass
            executor.shutdown(wait=False)

    processed_articles = [a for a in articles if a.get("status")]
    decode_ok = sum(1 for a in processed_articles if a.get("publisher_url"))
    alive_cnt = sum(1 for a in processed_articles if a.get("http_code") == 200)
    ok = sum(1 for a in processed_articles if " | ok |" in a.get("status", ""))
    total = sum(len(a.get("saved_images", [])) for a in processed_articles)

    xlsx = export_excel(processed_articles, output_dir)
    if interrupted:
        print(f"[crawl] safe exit (Ctrl+C): {len(processed_articles)}/{len(articles)} articles processed, {ok} with images -> {xlsx}")
    else:
        print(f"[crawl] done: {len(articles)} discovered, {decode_ok} decoded, {alive_cnt} alive 200, {ok} with images, {total} files -> {xlsx}")

    return {
        "keyword": keyword,
        "discovered": len(articles),
        "processed": len(processed_articles),
        "decoded": decode_ok,
        "alive": alive_cnt,
        "with_images": ok,
        "saved_images": total,
        "interrupted": interrupted,
        "xlsx": str(xlsx),
        "jsonl": str(jsonl_path),
        "output_dir": str(output_dir),
        "images_dir": str(images_dir),
    }


## 6. Demo lấy thông tin với từ khoá `Trillion` với tối đa là 5 News và xuất vào trong thư mục `output/`

In [ ]:
crawl_google_news(
    "Trillion",
    max_articles=100,
    output_dir="output",
    workers=10,
    proxy=None,
    rotate_ua=True,
    lang="vi",
    country="VN",
    timeout=10, # TIMEOUT cho MỘT request HTTP đơn lẻ
    resume=True,
    show_progress=True,
)


## 7. Kiểm tra `output/`

In [ ]:
from pathlib import Path

print("\n".join(str(p) for p in sorted(Path("output").rglob("*"))))
!ls -lh output/images | head -n 20
!ls -lh output/articles.xlsx

## 8. Preview Excel & ảnh inline

In [ ]:
import openpyxl

wb=openpyxl.load_workbook("output/articles.xlsx"); ws=wb.active
print(f"rows={ws.max_row} cols={ws.max_column} images={len(ws._images)}")
for r in ws.iter_rows(min_row=2, max_row=6, values_only=True):
    print(f"#{r[0]} {r[1][:40]:40} | {r[9]}")

In [ ]:
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
from pathlib import Path

imgs=sorted(Path("output/images").glob("article_*.jpg"))+sorted(Path("output/images").glob("article_*.webp"))+sorted(Path("output/images").glob("article_*.png"))
cols=4; rows=(len(imgs)+cols-1)//cols
plt.figure(figsize=(cols*3, rows*2.2))
processed_count = 0
for p in imgs:
    if processed_count >= 12:
        break
    try:
        img = Image.open(p)
        plt.subplot(rows,cols,processed_count+1); plt.imshow(img); plt.title(p.name,fontsize=7); plt.axis("off")
        processed_count += 1
    except UnidentifiedImageError:
        print(f"Warning: Could not identify image file {p.name}. Skipping.")
    except Exception as e:
        print(f"Warning: An error occurred while opening image file {p.name}: {e}. Skipping.")

plt.tight_layout(); plt.show()